# MoneyPrinterMerged - AI Video Generator

Combined features from MoneyPrinterTurbo + MoneyPrinterV2

**Features:**
- Auto video generation (script + voice + b-roll + subtitles + music)
- Twitter bot with scheduling
- YouTube Shorts automation
- Free b-roll sources (Pexels, Pixabay)
- Mistral AI integration

## Step 1: Clone and Install Dependencies

In [ ]:
# Clone repository
!git clone https://github.com/corruptcrew/KyxContent-Hub.git
%cd KyxContent-Hub

# Upgrade pip first
!pip install --upgrade pip -q

# Install ALL dependencies
!pip install streamlit==1.45.0 fastapi==0.115.6 uvicorn==0.32.1 -q
!pip install moviepy==2.1.2 edge-tts==7.2.7 imageio==2.37.3 imageio-ffmpeg==0.6.0 -q
!pip install openai==1.56.1 litellm==1.60.0 requests==2.33.1 pillow==10.4.0 -q
!pip install toml==0.10.2 loguru==0.7.3 pydantic==2.13.4 -q
!pip install aiohttp==3.13.5 httpx==0.27.2 python-dotenv==1.2.2 -q

# Install system packages
!apt-get update -y -qq
!apt-get install -y imagemagick ffmpeg -qq

# Fix ImageMagick policy
!sed -i 's/rights="none" pattern="@"/rights="read|write" pattern="@"/' /etc/ImageMagick-6/policy.xml 2>/dev/null || true

# Verify streamlit is installed
import streamlit
print(f'✅ Streamlit version: {streamlit.__version__}')
print('✅ Dependencies installed!')

## Step 2: Configure API Keys

In [ ]:
# Get FREE API keys:
# - Pexels: https://www.pexels.com/api/ (free, instant)
# - Pixabay: https://pixabay.com/api/ (free, instant)

PEXELS_API_KEY = ''  # Add your key here
PIXABAY_API_KEY = ''  # Optional
MISTRAL_API_KEY = '1frihpaCHeZ09oiXwE8cxLv2PlppFIkj'

config = '''[app]
video_source = "pexels"
pexels_api_keys = []
pixabay_api_keys = []
llm_provider = "openai"
openai_api_key = "1frihpaCHeZ09oiXwE8cxLv2PlppFIkj"
openai_base_url = "https://api.mistral.ai/v1"
openai_model_name = "mistral-small-latest"
subtitle_provider = "edge"
voice_name = "en-US-JennyNeural"
video_aspect = "portrait"
bgm_type = "random"
bgm_volume = 0.2
listen_host = "0.0.0.0"
listen_port = 8080
'''

with open('config.toml', 'w') as f:
    f.write(config)

print('✅ Configuration saved!')
print('Add your Pexels API key in config.toml for full video generation')

## Step 3: Start Ngrok Tunnel

In [ ]:
import subprocess
import time
import requests

# Kill any existing ngrok
subprocess.run(['pkill', '-9', 'ngrok'], stderr=subprocess.DEVNULL)

# Download ngrok
print('Downloading ngrok...')
!wget -q https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz -O /tmp/ngrok.tgz
!tar -xzf /tmp/ngrok.tgz -C /tmp

# Start tunnel
print('Starting ngrok tunnel to port 8501...')
subprocess.Popen(['/tmp/ngrok', 'http', '8501', '--log=stdout'])
time.sleep(5)

# Get public URL
try:
    response = requests.get('http://localhost:4040/api/tunnels', timeout=10)
    tunnel_url = response.json()['tunnels'][0]['public_url']
    print('')
    print('=' * 50)
    print('SUCCESS!')
    print('=' * 50)
    print('Open this URL in your browser:')
    print(tunnel_url)
    print('=' * 50)
    print('Keep this cell running while using the WebUI')
except Exception as e:
    print('Check ngrok dashboard: http://localhost:4040')

## Step 4: Start WebUI

In [ ]:
import subprocess
import time
import sys

# Start Streamlit using python -m (more reliable than !streamlit)
print('Starting Streamlit WebUI...')
!python -m streamlit run webui/Main.py --server.address=0.0.0.0 --server.port=8501 --server.headless true 2>&1 &

# Wait for startup
time.sleep(10)

# Verify it's running
result = subprocess.run(['lsof', '-i', ':8501'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ Streamlit WebUI is running on port 8501')
    print('Check the ngrok URL from previous cell to access it')
else:
    print('❌ Streamlit failed to start')
    print('Re-run Cell 1 to ensure dependencies are installed')
    print('Or try: !pip install streamlit==1.45.0')

## Alternative: Generate Video via API

In [ ]:
import requests
import time
from google.colab import files

# Start API server
!python main.py 2>&1 &
time.sleep(5)

API_BASE = 'http://localhost:8080'

payload = {
    'video_subject': 'Which is the best continent?',
    'video_aspect': 'portrait',
    'video_language': 'en',
    'voice_name': 'en-US-JennyNeural',
    'voice_volume': 1.0,
    'voice_rate': 1.0,
    'bgm_type': 'random',
    'bgm_volume': 0.2,
    'subtitle_enabled': True,
    'subtitle_provider': 'edge',
    'llm_provider': 'openai'
}

print('Generating video...')
response = requests.post(f'{API_BASE}/api/v1/videos', json=payload)
task_id = response.json()['data']['task_id']
print(f'Task ID: {task_id}')

while True:
    status = requests.get(f'{API_BASE}/api/v1/tasks/{task_id}')
    data = status.json()['data']
    print(f"Progress: {data['progress']}% - State: {data['state']}")
    
    if data['state'] == 1:
        print('Video generated!')
        video_url = f"{API_BASE}{data['video_url']}"
        print(f'Download: {video_url}')
        
        video_response = requests.get(video_url)
        with open('video.mp4', 'wb') as f:
            f.write(video_response.content)
        files.download('video.mp4')
        break
    elif data['state'] == -1:
        print(f"Failed: {data.get('reason', 'Unknown error')}")
        break
    
    time.sleep(5)